# DB9 Toolkit Training Local

Local workflow styled like DB9Studio Toolkit Training. Run cells from top to bottom.


# 1. Cai dat

In [ ]:
#@title ? Cai dat UI local

from pathlib import Path
import os
import subprocess
import sys

root_dir = Path.cwd()
if root_dir.name == "notebooks":
    os.chdir(root_dir.parent)
root_dir = Path.cwd()
toolkit_dir = root_dir / "ai-toolkit"

UI_Train = "Notebook" # @param ["Notebook", "Toolkit_UI"]
Run_Install = False #@param {type:"boolean"}

print(f"Project: {root_dir}")
print(f"ai-toolkit: {toolkit_dir}")

if Run_Install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])
    if not toolkit_dir.exists():
        subprocess.check_call(["git", "clone", "--recurse-submodules", "https://github.com/ostris/ai-toolkit.git", str(toolkit_dir)])
    subprocess.check_call(["git", "submodule", "update", "--init", "--recursive"], cwd=toolkit_dir)
    req = toolkit_dir / "requirements.txt"
    if req.exists():
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(req)])
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "setuptools<82"])
    subprocess.check_call([sys.executable, "verify.py"])
else:
    print("Skip install. Use setup_5090.bat or enable Run_Install if needed.")


# 2. Cau hinh train

In [ ]:
#@title ? 2.1 Xu ly du lieu

from pathlib import Path
import os
import subprocess
import sys
from dotenv import load_dotenv

load_dotenv('.env')

# @markdown ?? TrainFolder la thu muc anh goc
TrainFolder = "datasets/raw"  # @param {type:'string'}
# @markdown ?? Thu muc output anh/caption da xu ly
ProcessedFolder = "datasets/processed"  # @param {type:'string'}
ControlFolder = ""  # @param {type:'string'}
DataClean = True #@param {type:"boolean"}
No_gen_caption = False #@param {type:"boolean"}
Caption = "APIGemini | 2.5 Flash" # @param ["None", "APIGemini | 2.5 Flash", "APIGemini | 2.5 Pro", "APIGemini | 2.5 Flash Lite", "Florence"]
Caption_Length = "Medium" # @param ["Short", "Medium", "Long"]
API_Prompt = "Mo ta chi tiet hinh anh nay" # @param {type:'string'}
Custom_Caption = "" # @param {type:'string'}
Append = False #@param {type:"boolean"}
Overwrite_Caption = False #@param {type:"boolean"}
Min_Resolution = 512 #@param {type:'number'}
Max_Resolution = 4096 #@param {type:'number'}
Image_Format = "jpg" # @param ["jpg", "png", "webp"]

train_dir = Path(TrainFolder)
processed_dir = Path(ProcessedFolder)
img_dir = processed_dir / "img"
cap_dir = processed_dir / "captions"
img_dir.mkdir(parents=True, exist_ok=True)
cap_dir.mkdir(parents=True, exist_ok=True)

print(f"TrainFolder: {train_dir.resolve()}")
print(f"Processed images: {img_dir.resolve()}")
print(f"Captions: {cap_dir.resolve()}")

if DataClean:
    subprocess.check_call([
        sys.executable, "scripts/data_clean.py",
        "--input", str(train_dir),
        "--output", str(img_dir),
        "--min-res", str(Min_Resolution),
        "--max-res", str(Max_Resolution),
        "--format", Image_Format,
    ])
else:
    print("Skip data clean.")

if not No_gen_caption and Caption != "None":
    if Caption.startswith("APIGemini"):
        model_map = {
            "APIGemini | 2.5 Flash": "gemini-2.5-flash",
            "APIGemini | 2.5 Pro": "gemini-2.5-pro",
            "APIGemini | 2.5 Flash Lite": "gemini-2.5-flash-lite",
        }
        if not os.environ.get("GEMINI_API_KEY", "").strip():
            raise RuntimeError("Set GEMINI_API_KEY in .env before Gemini captioning.")
        cmd = [
            sys.executable, "scripts/caption_gemini.py",
            "--input", str(img_dir),
            "--output", str(cap_dir),
            "--model", model_map[Caption],
            "--length", Caption_Length,
            "--prompt", API_Prompt,
        ]
        if Custom_Caption:
            cmd += ["--custom-caption", Custom_Caption]
        if Append:
            cmd += ["--append"]
        if Overwrite_Caption:
            cmd += ["--overwrite"]
        subprocess.check_call(cmd)
    elif Caption == "Florence":
        subprocess.check_call([
            sys.executable, "scripts/caption_florence.py",
            "--input", str(img_dir),
            "--output", str(cap_dir),
            "--length", Caption_Length,
        ])
else:
    print("Skip caption generation.")

print(f"Images: {len(list(img_dir.glob('*')))}")
print(f"Captions: {len(list(cap_dir.glob('*.txt')))}")


In [ ]:
#@title ?? 2.1b Prepare Upscale Detail Dataset

from pathlib import Path
import subprocess
import sys

Use_Upscale_Detail_Tiles = False #@param {type:"boolean"}
Upscale_Input_Folder = "datasets/raw" #@param {type:'string'}
Upscale_Output_Folder = "datasets/processed" #@param {type:'string'}
Tile_Size = 2048 #@param {type:'number'}
Tile_Overlap = 256 #@param {type:'number'}
Max_Tiles_Per_Image = 0 #@param {type:'number'}
Min_Detail_Score = 3.0 #@param {type:'number'}
Degrade_Scale = 2.0 #@param {type:'number'}
Blur_Radius = 1.2 #@param {type:'number'}
Jpeg_Quality = 45 #@param {type:'number'}
Tile_Caption = "high detail upscale restoration, sharp architectural detail" #@param {type:'string'}
Overwrite_Tiles = False #@param {type:"boolean"}

if Use_Upscale_Detail_Tiles:
    cmd = [
        sys.executable, "scripts/prepare_upscale_detail_dataset.py",
        "--input", Upscale_Input_Folder,
        "--output", Upscale_Output_Folder,
        "--tile-size", str(Tile_Size),
        "--overlap", str(Tile_Overlap),
        "--max-tiles-per-image", str(Max_Tiles_Per_Image),
        "--min-detail-score", str(Min_Detail_Score),
        "--degrade-scale", str(Degrade_Scale),
        "--blur-radius", str(Blur_Radius),
        "--jpeg-quality", str(Jpeg_Quality),
        "--caption", Tile_Caption,
    ]
    if Overwrite_Tiles:
        cmd.append("--overwrite")
    subprocess.check_call(cmd)
    ProcessedFolder = Upscale_Output_Folder
    ControlFolder = str(Path(Upscale_Output_Folder) / "control")
    print(f"Upscale detail dataset ready: {Upscale_Output_Folder}")
    print(f"ControlFolder: {ControlFolder}")
else:
    print("Skip upscale detail tiling. Enable Use_Upscale_Detail_Tiles for 2048 tile pairs.")


In [ ]:
#@title ?? 2.2 Cai dat train

import os
import sys
from pathlib import Path
sys.path.append('scripts')
from config_generator import generate_config

TypeTrain = "FLUX.2-klein-base-9B" # @param ["FLUX.2-klein-base-9B", "Flux"]
Training_Mode = "text2img" # @param ["text2img", "img2img_upscale"]
Low_VRAM = False #@param {type:"boolean"}
Lora_name = "db9_toolkit_trainner" # @param {type:"string"}
OutputFolder = "outputs" #@param {type:'string'}
Steps = 2000 #@param {type:'number'}
Save_steps = 500 #@param {type:'number'}
Sample_steps = 500 #@param {type:'number'}
Resolution = "1536" #@param {type:'string'}
Batch_size = 6 #@param {type:'number'}
Gradient_Accumulation = 1 #@param {type:'number'}
Lr = 4e-4 #@param {type:'number'}
Lr_Scheduler = "constant_with_warmup" # @param ["constant", "constant_with_warmup", "cosine", "linear", "cosine_with_restarts", "polynomial"]
Warmup_steps = 100 #@param {type:'number'}
Dim = 64 #@param {type:'number'}
Alpha = 64 #@param {type:'number'}
Optimizer = "adamw8bit" # @param ["adamw8bit", "adam", "adamw"]
Gradient_Checkpointing = True #@param {type:"boolean"}
Quantize = False #@param {type:"boolean"}
Enable_Bucketing = True #@param {type:"boolean"}
Bucket_step = 64 #@param {type:'number'}
Min_Bucket_Reso = 512 #@param {type:'number'}
Max_Bucket_Reso = 2048 #@param {type:'number'}
Flip_Aug = False #@param {type:"boolean"}
Color_Aug = False #@param {type:"boolean"}
Sample_Prompt = "A portrait of a person in DB9 style, highly detailed" #@param {type:'string'}

model_path = "black-forest-labs/FLUX.2-klein-base-9B" if TypeTrain == "FLUX.2-klein-base-9B" else "black-forest-labs/FLUX.1-dev"
resolution = int(str(Resolution).split(',')[0].strip())

if Low_VRAM:
    Batch_size = min(Batch_size, 1)
    resolution = min(resolution, 1024)
    Gradient_Checkpointing = True

base_dir = Path.cwd()
dataset_path = str((base_dir / ProcessedFolder).resolve()).replace("\\", "/")
output_path = str((base_dir / OutputFolder).resolve()).replace("\\", "/")
control_folder_path = ""
if Training_Mode == "img2img_upscale":
    control_folder_path = str((base_dir / ControlFolder).resolve()).replace("\\", "/")
    if not Path(ControlFolder).exists():
        raise FileNotFoundError(f"ControlFolder not found: {ControlFolder}. Run 2.1b upscale detail tiling first.")

config_yaml = generate_config(
    project_name=Lora_name,
    model_path=model_path,
    dataset_path=dataset_path,
    output_path=output_path,
    lora_rank=Dim,
    lora_alpha=Alpha,
    batch_size=Batch_size,
    learning_rate=Lr,
    train_steps=Steps,
    gradient_accumulation=Gradient_Accumulation,
    resolution=resolution,
    enable_bucketing=Enable_Bucketing,
    bucket_step=Bucket_step,
    min_bucket_reso=Min_Bucket_Reso,
    max_bucket_reso=Max_Bucket_Reso,
    flip_aug=Flip_Aug,
    color_aug=Color_Aug,
    optimizer=Optimizer,
    lr_scheduler=Lr_Scheduler,
    warmup_steps=Warmup_steps,
    gradient_checkpointing=Gradient_Checkpointing,
    quantize=Quantize,
    save_every=Save_steps,
    sample_every=Sample_steps,
    sample_prompts=[Sample_Prompt] if Sample_Prompt else [],
    training_mode=Training_Mode,
    control_folder_path=control_folder_path,
)

Path("configs").mkdir(exist_ok=True)
config_path = Path("configs") / f"{Lora_name}.yaml"
config_path.write_text(config_yaml, encoding="utf-8")
print(f"Config saved: {config_path}")
print(f"Resolution: {resolution}px")
print(f"Output: {output_path}/{Lora_name}")
print(f"Training mode: {Training_Mode}")
if control_folder_path:
    print(f"Control: {control_folder_path}")


# 3. Train

In [ ]:
#@title ? Run Lora Train

from pathlib import Path
import subprocess
import sys

AutoOpenOutput = True #@param {type:"boolean"}
RunTrain = False #@param {type:"boolean"}

config_path = Path("configs") / f"{Lora_name}.yaml"
toolkit_run = Path("ai-toolkit") / "run.py"

print(f"Config: {config_path}")
print(f"Toolkit: {toolkit_run}")

if not config_path.exists():
    raise FileNotFoundError("Missing config. Run cell 2.2 first.")
if not toolkit_run.exists():
    raise FileNotFoundError("Missing ai-toolkit/run.py. Run setup_5090.bat first.")

if RunTrain:
    subprocess.check_call([sys.executable, "run.py", "../" + config_path.as_posix()], cwd="ai-toolkit")
else:
    print("RunTrain is False. Set RunTrain=True to start training.")
    print(f"Manual command: cd ai-toolkit && {sys.executable} run.py ../{config_path.as_posix()}")

if AutoOpenOutput:
    out = Path(OutputFolder) / Lora_name
    out.mkdir(parents=True, exist_ok=True)
    print(f"Output folder: {out.resolve()}")


In [ ]:
#@title ?? Test LoRA checkpoint

from pathlib import Path
import os

RunTest = False #@param {type:"boolean"}
TestPrompt = "A portrait of a person in DB9 style, highly detailed" #@param {type:'string'}

checkpoint_dir = Path(OutputFolder) / Lora_name
checkpoints = sorted(checkpoint_dir.rglob("*.safetensors"), key=lambda p: p.stat().st_mtime)
print(f"Checkpoint folder: {checkpoint_dir.resolve()}")
print(f"Found checkpoints: {len(checkpoints)}")
if checkpoints:
    print(f"Newest: {checkpoints[-1]}")

if RunTest:
    import torch
    from diffusers import FluxPipeline
    if not checkpoints:
        raise FileNotFoundError("No checkpoint found. Train first.")
    pipe = FluxPipeline.from_pretrained(model_path, torch_dtype=torch.bfloat16)
    pipe.load_lora_weights(str(checkpoints[-1]))
    pipe.to("cuda")
    image = pipe(TestPrompt, num_inference_steps=28, guidance_scale=1.0, height=resolution, width=resolution).images[0]
    Path("outputs/test").mkdir(parents=True, exist_ok=True)
    image.save("outputs/test/test_result.png")
    print("Saved outputs/test/test_result.png")
    image
else:
    print("RunTest is False. Set RunTest=True after training.")
